
# Beyond the Weekday — Reproducibility Notebook

**Beyond the Weekday: An Open-Data Framework for Holiday Transit Equity, with an Application to Milan**

Amirhassan Darvishzade · MSc Mobility Engineering, Politecnico di Milano · 2025–2026  
Advisor: Prof. Fabio Borghetti · Co-advisor: Alberto Bertolin

This notebook supports reproduction of the five-phase open-data framework using GTFS, small-area census data and OpenStreetMap. The Milan application uses ISTAT census data and archived transit and essential-service datasets.

Optional operator-data extensions assess mode and depot constraints using ATM data. These private datasets are not redistributed with the repository.

### How to run

1. Place this notebook in the repository root, with the required inputs under `data/` as declared in `config.py`.
2. Install the dependencies listed in `requirements.txt`.
3. Check the paths and parameters in `config.py`.
4. Restart the Jupyter kernel and select **Run All** to execute the workflow.
5. Inspect the generated results and run records in `outputs/`.

See `README.md` for the run order, data sources, known issues and the phase-numbering note.

**Reproducibility note:** Use the archived input datasets, including the original essential-service snapshot. Querying current OpenStreetMap data may produce different results. Operator-data extensions require separately authorised ATM inputs.


## 0 · Setup and pre-flight checks

In [ ]:
import os, sys, platform, warnings

NOTEBOOK_BUILD = "2026-10-28-release-1.0"
print("Notebook build:", NOTEBOOK_BUILD)
from pathlib import Path

PROJECT_DIR = Path(os.environ.get("OPEN_DATA_FRAMEWORK_DIR", Path.cwd())).resolve()
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
warnings.filterwarnings("ignore", category=FutureWarning)
print("Working folder:", PROJECT_DIR)

CORE_REQUIRED = ["config.py", "build_stop_table.py", "optimise.py", "sweep.py", "qgis_layers.py",
                 "hour_diff.py", "phase1_corridors.py", "phase2_holiday_disaggregation.py",
                 "phase3_optimisation_map.py", "phase4_essential_services.py",
                 "sensitivity_dynamic_burden.py", "sensitivity_suite.py", "robustness_checks.py"]
missing = [f for f in CORE_REQUIRED if not (PROJECT_DIR / f).exists()]
assert not missing, f"Missing core modules: {missing}"

import config
assert getattr(config, "FRAMEWORK_VERSION", None) == "1.1",     f"Expected framework version 1.1, found {getattr(config, 'FRAMEWORK_VERSION', None)}"
config.validate()

import numpy as np
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 100)
OUT = config.OUTPUT_DIR
print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
print("Core modules present — ready to run.")

In [ ]:
# Configuration used by every phase
print("config file:          ", config.__file__)
print("GTFS feeds:           ", list(config.GTFS_FEEDS))
print("core holiday dates:   ", config.CORE_HOLIDAY_DATES)
print("all holiday dates:    ", sorted(config.ALL_HOLIDAY_DATES))
print("operational window:    [%d, %d) -> %d min" % (config.OPERATIONAL_HOUR_START,
                                                    config.OPERATIONAL_HOUR_END,
                                                    config.OPERATIONAL_MINUTES))
print("overflow rule:        ", config.OVERFLOW_RULE)
print("scenarios:            ", config.SCENARIOS)
print("cross-holiday review: ", config.CROSS_HOLIDAY_REVIEW)
print("config.validate():    ", config.validate())

## 1 · Phases 1–2: vulnerability composite and scheduled supply
Builds the stop-hourly table: vulnerability attached to stops, reconstructed calendars, day-type headways and social burden (§3.2–3.3). Heavy step: full GTFS parse.

In [ ]:
import build_stop_table
build_stop_table.main()

## 2 · Phase 5: named scenarios and the continuous sweep
Budget-neutral stop-level reallocation with the 8-minute recipient cap (§3.6, §5.5.2), then the sweep of donor-reduction rates from 5% to 70% that locates the plateau and its numerical peak (§5.5.3).

In [ ]:
import optimise
optimise.main()

In [ ]:
import sweep
sweep.main()          # heavy: 66 optimiser runs

In [ ]:
# Overflow-rule robustness. The principal thesis curve keeps the configured
# redistribute-then-cap rule. If high rates leave residual service unallocated,
# a separate residual-return sweep is written under distinct filenames.
curve = pd.read_csv(os.path.join(OUT, "sweep_curve.csv"))
max_unalloc = curve["unallocated_trips"].max()
print(f"Maximum unallocated equivalents across principal sweep: {max_unalloc:.2f}")

if max_unalloc >= config.BUDGET_NEUTRALITY_TOLERANCE:
    sweep.main(overflow_rule="return_residual_to_donors", output_tag="residual_return")
    print("Residual-return robustness sweep written separately; principal files were not overwritten.")
else:
    print("No residual appears in the swept range; no second sweep is needed.")

## 3 · Within-day pattern, service floor and corridors of exclusion
Map layers, hourly profiles and holiday-versus-Sunday deterioration (§5.2.1), then the 30-minute service floor and the ranking of the most burdened stops (§5.2).

In [ ]:
import qgis_layers
qgis_layers.main()

In [ ]:
# Per-stop hour-to-hour differences (Figure 5.7). The window is [6, 22): valid hours are 6..21.
import hour_diff
hour_diff.main()               # 08:00 vs 17:00
hour_diff.main(a=8, b=12)
hour_diff.main(a=12, b=21)
hour_diff.main(a=8, b=21)

In [ ]:
import phase1_corridors
phase1_corridors.main()

## 4 · Reallocation roles and before/after maps (Phase 5, §5.5)

In [ ]:
import phase3_optimisation_map
phase3_optimisation_map.main()

## 5 · Phase 3: per-holiday comparison
Each holiday recomputed separately across the four seasonal feeds; Labour Day is reported but excluded from the ranking because of its GTFS coverage collapse (§3.4, §5.3). Requires section 3.

In [ ]:
import phase2_holiday_disaggregation
phase2_holiday_disaggregation.main()

## 6 · Phase 4: essential access and double exclusion
Constraint-limited walking access to healthcare, pharmacies and food retail, crossed with top-quartile holiday burden (§3.5, §5.4). Requires sections 2 and 4.

In [ ]:
import phase4_essential_services
phase4_essential_services.main()

In [ ]:
# Phase 4 already constructs burden with the disclosed 120-minute no-service
# convention before classifying double exclusion. No duplicate served-vs-penalised
# recomputation is needed here.
import geopandas as gpd
p4 = gpd.read_file(os.path.join(OUT, "phase4_essential_access.gpkg"))
print("Phase 4 stops:", len(p4), "| double-excluded:", int(p4["double_exclusion"].astype(bool).sum()))

## 7 · Static versus dynamic burden weights (§5.5.4)

In [ ]:
import sensitivity_dynamic_burden
sensitivity_dynamic_burden.main()

## 8 · Optional operator-data extensions (§5.6)
These cells run only when the non-public ATM workbook / depot registry is present. Their absence does not prevent reproduction of §§5.1–§5.5.

In [ ]:
import depot_feasibility
if os.path.exists(depot_feasibility.ATM_WORKBOOK):
    depot_feasibility.main()
else:
    print("Skipped depot feasibility screen: private ATM workbook not present.")

In [ ]:
import demand_validation
atm_book = getattr(demand_validation, "ATM_WORKBOOK", None)
if atm_book and os.path.exists(atm_book):
    demand_validation.main()
else:
    print("Skipped metro-entry consistency check: private ATM workbook not present.")

In [ ]:
# Inspect the private line-to-depot registry only when available.
if os.path.exists(depot_feasibility.ATM_WORKBOOK):
    reg = pd.read_excel(depot_feasibility.ATM_WORKBOOK, sheet_name=depot_feasibility.DEPOT_SHEET)
    reg.columns = [str(c).strip() for c in reg.columns]
    reg = reg.rename(columns={depot_feasibility.COL_LINE: "line", depot_feasibility.COL_DEPOT: "depot",
                              depot_feasibility.COL_AMBITO: "ambito"})
    reg["line"] = reg["line"].astype(str).str.strip().str.upper()
    reg = reg.dropna(subset=["line", "depot"]).drop_duplicates(["line", "depot"])
    multi = reg["line"].value_counts(); multi = multi[multi > 1]
    print(f"register rows {len(reg)} | unique lines {reg['line'].nunique()} | multi-depot lines {len(multi)}")
    print(reg.drop_duplicates("line")["ambito"].value_counts().to_string())

### 8a · Depot- and mode-constrained re-run (§5.6.3)
When the private line-to-depot registry is available, this cell reproduces the mode-only, 2,000-equivalent allowance and strict regimes. Otherwise it is skipped.

In [ ]:
import depot_constrained_optimise as depot_opt
if os.path.exists(depot_opt.DEPOT_REGISTRY):
    depot_opt.main()
else:
    print("Skipped §5.6.3 constrained re-run: private depot registry not present.")

## 9 · Sensitivity register (§5.5.4)
Weight simplex, service-floor thresholds, tertile cuts, catchments, continuous priority, bridge-day relaxation and seeded bootstrap envelopes.

In [ ]:
import sensitivity_suite
sensitivity_suite.main()

## 10 · Consolidated robustness checks
`robustness_checks.py` replaces the three historical `review_checks*.py` scripts. It contains the thesis robustness/diagnostic checks in one deterministic entry point.

In [ ]:
import subprocess

def run_script(script, *args, log=None):
    cmd = [sys.executable, script, *args]
    p = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8",
                       env=dict(os.environ, PYTHONIOENCODING="utf-8"))
    if log:
        with open(log, "w", encoding="utf-8") as f:
            f.write(p.stdout + ("\n\n--- STDERR ---\n" + p.stderr if p.stderr else ""))
    if p.returncode != 0:
        print(f"{script} failed (return code {p.returncode}):\n{p.stderr[-3000:]}")
    return p.stdout

# Check 10 needs the private ATM workbook; it is skipped when the workbook is absent.
# The log is written to outputs/ (not committed), since it can contain ATM figures.
robust_args = ["--skip-osm"]
if not os.path.exists(config.DEPOT_REGISTRY):
    robust_args.append("--skip-atm")
robust = run_script("robustness_checks.py", *robust_args,
                    log=os.path.join(OUT, "robustness_output.txt"))
print(robust[-12000:])

In [ ]:
# The OSM boundary-buffer check is intentionally optional because it downloads
# current OSM data rather than the frozen July-2026 snapshot.
print("Run `python robustness_checks.py --checks 9` separately if you want the current-OSM boundary test.")

In [ ]:
# Direction-pooling sensitivity (§6.3.2) is produced by robustness check 8.
direction_path = os.path.join(OUT, "robustness_08_direction_sensitivity.csv")
if os.path.exists(direction_path):
    direction_rows = pd.read_csv(direction_path)
    print(direction_rows.round(3).to_string(index=False))
    print("\nThe published_legacy row reproduces the thesis-era 18.8% sensitivity.")
    print("The corrected_consistent row recomputes holiday service equivalents after direction adjustment.")
else:
    direction_rows = pd.DataFrame()
    print("Direction-sensitivity output unavailable; check robustness output for check 8.")


In [ ]:
from functools import lru_cache
checks = []

@lru_cache(maxsize=None)
def rd(name, index_col=None):
    return pd.read_csv(os.path.join(OUT, name), index_col=index_col)

def check(label, where, expected, decimals, compute):
    try:
        value = float(compute())
        tol = 0.5 * 10 ** (-decimals)
        ok = abs(value - float(expected)) < tol + 1e-12
        checks.append({"figure": label, "thesis": where, "expected": expected,
                       "run": value, "status": "OK" if ok else "DIFF"})
    except Exception as e:
        checks.append({"figure": label, "thesis": where, "expected": expected,
                       "run": None, "status": f"n/a ({type(e).__name__}: {e})"[:100]})

def hourly():
    h = rd("milan_stop_hourly.csv")
    return h[config.operational_hour_filter(h["hour"])]

def opt():
    return rd("milan_optimised_moderate.csv")

def weekday_stop_burden():
    o = opt()
    return (o["v_index"] * o["hw_wd"]).mean()

def pooled_by_role(role):
    d = rd("robustness_08_direction_pooling.csv")
    pooled = (d["both_same_route"].fillna(False).astype(bool)
              | d["metro_station"].fillna(False).astype(bool))
    return int((pooled & (d["role"] == role)).sum())

def one_direction_gini(implementation):
    d = rd("robustness_08_direction_sensitivity.csv")
    if pd.api.types.is_bool_dtype(d["dynamic_burden"]):
        dynamic = d["dynamic_burden"]
    else:
        dynamic = d["dynamic_burden"].astype(str).str.lower().isin(["true", "1", "yes"])
    row = d[(d["headways"] == "one direction per route")
            & (d["implementation"] == implementation) & ~dynamic]
    if row.empty:
        raise ValueError(f"one-direction static row not found: {implementation}")
    return row["gini_improvement_%"].iloc[0]

def retention():
    return rd("demand_vs_supply_station.csv")[["sup_ret_hol", "dem_ret_hol"]].dropna()

# Phase 4 checks use the frozen July-2026 service layer when it is present.
p4_check = gpd.read_file(os.path.join(OUT, "phase4_essential_access.gpkg"))
tertile_check = pd.qcut(p4_check["v_index"], 3, labels=["low", "middle", "high"])
de = p4_check["double_exclusion"].astype(bool)
de_rate = de.groupby(tertile_check, observed=False).mean() * 100

access_ok = p4_check["essential_access"]
if not pd.api.types.is_bool_dtype(access_ok):
    access_ok = access_ok.astype(str).str.lower().isin(["true", "1", "yes"])
strict20 = (p4_check["hw_hol"] > 20) & ~access_ok
strict20_rate = strict20.groupby(tertile_check, observed=False).mean() * 100

check("analytical stops", "§4.3.5", 3187, 0, lambda: len(opt()))
check("holiday/weekday burden, served", "§5.2", 1.49, 2,
      lambda: hourly()["burden_hol"].mean() / hourly()["burden_wd"].mean())
check("holiday/weekday burden, penalised", "§5.2", 1.62, 2,
      lambda: opt()["burden_hol"].mean() / weekday_stop_burden())
check("penalised ratio after Moderate", "§5.5.2", 1.406, 3,
      lambda: opt()["burden_hol_opt"].mean() / weekday_stop_burden())

check("double-excluded stops", "§5.4", 415, 0, lambda: de.sum())
for t, v in (("low", 6.8), ("middle", 10.2), ("high", 22.1)):
    check(f"double exclusion, {t} tertile (%)", "§5.4", v, 1, lambda t=t: de_rate[t])
check("priority stops", "§5.4", 184, 0, lambda: (de & (p4_check["role"] == "recipient")).sum())
check("access failure, low tertile (%)", "Table 5.3a", 53.0, 1,
      lambda: (~access_ok).groupby(tertile_check, observed=False).mean()["low"] * 100)
check("access failure, high tertile (%)", "Table 5.3a", 42.1, 1,
      lambda: (~access_ok).groupby(tertile_check, observed=False).mean()["high"] * 100)
for t, v in (("low", 11.4), ("middle", 5.5), ("high", 9.1)):
    check(f"headway >20 and poor access, {t} (%)", "§5.4", v, 1,
          lambda t=t: strict20_rate[t])

scen = lambda: rd("optimise_scenarios.csv", "scenario")
check("donor stops", "§5.5.1", 336, 0, lambda: (p4_check["role"] == "donor").sum())
check("recipient stops", "§5.5.1", 361, 0, lambda: (p4_check["role"] == "recipient").sum())
check("Moderate Gini improvement (%)", "§5.5.2", 20.8, 1, lambda: scen().loc["Moderate", "gini_improvement_%"])
check("Moderate burden reduction (%)", "§5.5.2", 13.2, 1, lambda: scen().loc["Moderate", "burden_reduction_%"])
check("Moderate equivalents moved/day", "§5.5.2", 10015, 0, lambda: scen().loc["Moderate", "freed_trips"])
check("sweep peak Gini (%)", "§5.5.3", 22.9, 1,
      lambda: rd("sweep_curve.csv").query("budget_neutral")["gini_improvement_%"].max())
check("sweep peak rate", "§5.5.3", 0.41, 2,
      lambda: rd("sweep_curve.csv").query("budget_neutral").set_index("rate")["gini_improvement_%"].idxmax())

check("pooled-headway donors", "§6.3.2", 81, 0, lambda: pooled_by_role("donor"))
check("pooled-headway recipients", "§6.3.2", 16, 0, lambda: pooled_by_role("recipient"))

print("Direction sensitivity used for thesis check: published_legacy =",
      one_direction_gini("published_legacy"))
print("Corrected post-thesis direction sensitivity =",
      one_direction_gini("corrected_consistent"))
check("one-direction Moderate Gini (%)", "§6.3.2", 18.8, 1,
      lambda: one_direction_gini("published_legacy"))

check("metro stations compared", "§5.6.1", 108, 0, lambda: len(retention()))
check("supply vs entry retention Pearson r", "§5.6.1", -0.01, 2, lambda: retention().corr().iloc[0, 1])
check("median holiday supply retention", "§5.6.1", 0.722, 3, lambda: retention()["sup_ret_hol"].median())
check("median holiday entry retention", "§5.6.1", 0.485, 3, lambda: retention()["dem_ret_hol"].median())
check("Area Urbana equivalents gained", "§5.6.2", 2138, 0,
      lambda: rd("depot_ambito_balance.csv", "ambito").loc["Area Urbana", "gained"])
check("Area Urbana equivalents freed", "§5.6.2", 540, 0,
      lambda: rd("depot_ambito_balance.csv", "ambito").loc["Area Urbana", "freed"])

check("depot mode-only Moderate Gini (%)", "§5.6.3", 15.9, 1,
      lambda: rd("depot_constrained_scenarios.csv").query("scenario == 'Moderate'")["gini_free_%"].iloc[0])
check("depot allowance Moderate Gini (%)", "§5.6.3", 14.5, 1,
      lambda: rd("depot_constrained_scenarios.csv").query("scenario == 'Moderate'")["gini_allowance_%"].iloc[0])
check("depot strict Moderate Gini (%)", "§5.6.3", 10.2, 1,
      lambda: rd("depot_constrained_scenarios.csv").query("scenario == 'Moderate'")["gini_strict_%"].iloc[0])
check("strict equivalents placed", "§5.6.3", 5571, 0,
      lambda: rd("depot_constrained_scenarios.csv").query("scenario == 'Moderate'")["reallocated_strict"].iloc[0])
check("strict equivalents returned", "§5.6.3", 1959, 0,
      lambda: rd("depot_constrained_scenarios.csv").query("scenario == 'Moderate'")["returned_strict"].iloc[0])

report = pd.DataFrame(checks)
report.to_csv(os.path.join(OUT, "thesis_reproduction_check.csv"), index=False)
print(report.to_string(index=False))
n_ok = int((report["status"] == "OK").sum())
n_diff = int((report["status"] == "DIFF").sum())
n_na = len(report) - n_ok - n_diff
print(f"\nReproduction summary: {n_ok}/{len(report)} OK, {n_diff} DIFF, {n_na} n/a.")


try:
    corrected_direction_gini = one_direction_gini("corrected_consistent")
    print(f"Corrected post-thesis one-direction Moderate Gini: {corrected_direction_gini:.3f}% ")
    print("(reported separately; not included in the thesis reproduction pass/fail count).")
except Exception as e:
    print(f"Corrected one-direction diagnostic unavailable: {e}")
